# block-group-stack — worked example 1: Build a BlockGroup and report its per-block feature widths

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `block-group-stack`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A ResNet **BlockGroup** is an `nn.Sequential` of `N` blocks where only block 0 changes shape: it maps `in_feats -> out_feats` and applies `first_stride`. Every later block runs at `(out_feats, out_feats, stride=1)`, preserving shape. The asymmetry means exactly one block downsamples; the rest are identity-shaped.

## Worked solution

We want to both *build* the group and *describe* it.

**Step 1 - block 0 is special.** It is the only block that crosses channel counts and carries the stride: `ResBlock(in_feats, out_feats, first_stride=first_stride)`. This is where any downsampling happens.

**Step 2 - the remaining `n_blocks - 1` blocks are uniform.** Each is `ResBlock(out_feats, out_feats, first_stride=1)`. They cannot change channels (in == out) and cannot downsample (stride 1), so they preserve the tensor shape exactly. We append them in a loop.

**Step 3 - wrap in `nn.Sequential(*blocks)`.** Splatting the list makes each block a positional child so `group[i]` indexes them.

**Step 4 - report widths.** Reading `b.out_feats` off each child gives `[out_feats]*n_blocks` for a canonical group. Because every block after block 0 keeps `out_feats`, the width list is constant after index 0, which is the invariant we expect. Printing it confirms the build is correct.

In [ ]:
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)

def make_group_and_widths(in_feats, out_feats, n_blocks, first_stride):
    blocks = [ResBlock(in_feats, out_feats, first_stride=first_stride)]
    for _ in range(n_blocks - 1):
        blocks.append(ResBlock(out_feats, out_feats, first_stride=1))
    group = nn.Sequential(*blocks)
    widths = [b.out_feats for b in group]
    return group, widths

group, widths = make_group_and_widths(in_feats=16, out_feats=32, n_blocks=4, first_stride=2)
print('n_blocks:', len(group))
print('block0 stride:', group[0].first_stride, 'block0 in->out:', group[0].in_feats, '->', group[0].out_feats)
print('widths:', widths)